In [2]:
import ast
import os

In [3]:
from urllib.parse import urlparse

def get_filename(url: str):
    if url[-4:] == ".git":
        url = url[:-4]
        
    parts = urlparse(url).path.strip("/").split("/")

    if len(parts) >= 2:
        username = parts[0]
        project_name = parts[1]
        result = f"{username}-{project_name}"
        return result

In [4]:
repo = "https://github.com/princ3kr/Notebook-LM-Mini"

filename = get_filename(repo)
dir = f"../temp/{filename}"

if os.path.isdir(dir):
    print("Directory already exists")

Directory already exists


In [5]:
import subprocess

if os.path.isdir(dir):
    print("Directory already exists")
else:
    try:
        subprocess.run(["git", "clone", repo, dir], check=True)
        print("Clone successful!")
    except subprocess.CalledProcessError as e:
        print(f"Git command failed with error: {e}")


Directory already exists


In [6]:
# tree = ast.parse(files["src\\services\\tutor_service.py"])
# print(ast.dump(tree, indent=2))

def parse_file(source_code):
    tree = ast.parse(source_code)
    import_modules, import_names, classes, functions = [], [], [], []
    
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom):
            import_modules.append(node.module)
            import_names.append([alias.name for alias in node.names])

        if isinstance(node, ast.ClassDef):
            classes.append({
                "name": node.name,
                "line_start": node.lineno,
                "line_end": node.end_lineno
            })
            
        if isinstance(node, ast.FunctionDef):
            functions.append({
                "name": node.name,
                "line_start": node.lineno,
                "line_end": node.end_lineno
            })
                
    return {"import_modules": import_modules, "import_names": import_names, "classes": classes, "functions": functions}

In [7]:
ignores = { ".git", ".gitignore", ".lock", ".venv", "__pycache__", "node_modules", ".vscode", "pyproject.toml", "__init__.py", ".python-version", "requirements.txt" }

def get_structure(directory: str):
    for root, dirs, files in os.walk(directory):
        dirs[:] = sorted([d for d in dirs if d not in ignores])

        rel_dir = os.path.relpath(root, directory)
        
        if rel_dir == ".":
            level = 0
            display_name = os.path.basename(os.path.abspath(directory))
        else:
            level = rel_dir.count(os.sep) + 1
            display_name = os.path.basename(root)

        indent = "│   " * (level - 1) if level > 0 else ""
        connector = "├── " if level > 0 else ""
        print(f"{indent}{connector}{display_name}/")

        file_indent = "│   " * level
        sorted_files = sorted(files)
        for i, name in enumerate(sorted_files):
            if name not in ignores or name == "README.md":
                file_connector = "└── " if (i == len(sorted_files) - 1 and not dirs) else "├── "
                print(f"{file_indent}{file_connector}{name}")


def get_files(directory: str):
    inventory = {}
    for root, dirs, files in os.walk(directory):
        rel_dir = os.path.relpath(root, directory)
        if rel_dir == ".":
            rel_dir = ""
        
        for name in files:
            if (name not in ignores) and (name == "README.md" or name.endswith(".py")):
                full_path = os.path.join(root, name)

                with open(full_path, "r", encoding="utf-8") as f:
                    content = f.read()

                    parsed_structure = {"import_modules": [], "import_names": [], "classes": [], "functions": []}

                    if name.endswith(".py"):
                        parsed_structure = parse_file(content)

                    parsed_structure['content'] = content

                    inventory[os.path.join(rel_dir, name)] = parsed_structure
                        

    return inventory

In [8]:
# get_structure(dir)
files = get_files(directory=dir)

In [9]:
for imports in files['src\services\diagnoser_service.py']:
    print(imports)

import_modules
import_names
classes
functions
content


<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\PRINCE KUMAR\AppData\Local\Temp\ipykernel_16232\1899480365.py:1: SyntaxWarning: invalid escape sequence '\s'
  for imports in files['src\services\diagnoser_service.py']:


In [10]:
import networkx as nx
from pyvis.network import Network
from neo4j import GraphDatabase
import json

class BuildGraph:
    def __init__(self, files: dict):
        self.G = nx.DiGraph()
        self.files = {filepath.replace("\\", "/"): data for filepath, data in files.items()}
        self.name_index = self.build_name_index()

    def build_name_index(self) -> dict:
        name_index = {}
        for filepath in self.files:
            for classname in self.files.get(filepath, {}).get("classes", []):
                name_index[classname['name']] = filepath
            for funcname in self.files.get(filepath, {}).get("functions", []):
                name_index[funcname['name']] = filepath
        return name_index

    def get_path(self) -> dict:
        cleared_files = {}
        for filepath in self.files:
            imports = []
            modules = self.files[filepath].get("import_modules", [])
            names = self.files[filepath].get("import_names", [])

            for imp, imp_names in zip(modules, names):
                if not imp or not imp.startswith("src."): ## harcoded!
                    continue

                target = imp.replace(".", "/") + ".py" ## hardcoded!
                if target in self.files:
                    imports.append(target)
                else:
                    for name in imp_names:
                        if name in self.name_index:
                            imports.append(self.name_index[name])
                            
            cleared_files[filepath] = imports
        return cleared_files

    def get_node_style(self, filepath):
        """Returns a more premium color palette and node properties."""
        if "services" in filepath:
            color = {"background": "#3b82f6", "border": "#60a5fa", "highlight": "#93c5fd"}
        elif "models" in filepath:
            color = {"background": "#10b981", "border": "#34d399", "highlight": "#6ee7b7"}
        elif "database" in filepath:
            color = {"background": "#f59e0b", "border": "#fbbf24", "highlight": "#fcd34d"}
        else:
            color = {"background": "#8b5cf6", "border": "#a78bfa", "highlight": "#c4b5fd"}
        
        return color

    def build(self):
        cleared_path = self.get_path()
        for source, targets in cleared_path.items():
            style = self.get_node_style(source)
            self.G.add_node(
                source, 
                label=source.split("/")[-1],
                title=source,
                color=style,
                shadow=True,
                shape="dot",
                size=25,
                font={"color": "white", "size": 14, "face": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif"}
            )
            for target in targets:
                self.G.add_edge(source, target)

    def push_to_neo4j(self, uri, user, password):
        driver = GraphDatabase.driver(
            uri, auth=(user, password)
        )
        
        repo_id = filename

        with driver.session() as session:
            for node in self.G.nodes():
                imports = [
                    name for names in self.files[node]['import_names'] 
                    for name in names if name in self.name_index
                ]

                query = '''
                    MERGE (r:Repo {repo_id: $repo_id})
                    MERGE (f:File {path: $path, repo_id: $repo_id})
                    MERGE (r)-[:CONTAINS]->(f)
                    SET f.name = $name,
                        f.classes = $classes,
                        f.imports = $imports,
                        f.functions = $functions
                '''
                session.run(
                    query,
                    name=node.split('/')[-1],
                    path=node,
                    classes=[c['name'] for c in self.files.get(node, {}).get("classes", [])],
                    imports=imports,
                    functions=[f['name'] for f in self.files.get(node, {}).get("functions", [])],
                    repo_id=repo_id
                )
                
            for source, target in self.G.edges():
                query = '''
                    MATCH (a:File {path: $source, repo_id: $repo_id})
                    MATCH (b:File {path: $target, repo_id: $repo_id})
                    MERGE (a)-[:IMPORTS]->(b)
                '''
                session.run(query, source=source, target=target, repo_id=repo_id)
                

    def show(self, filename="graph.html"):
        net = Network(
            directed=True, 
            notebook=True, 
            cdn_resources='remote', 
            height="750px", 
            width="100%", 
            bgcolor="#111827",
            font_color="white"
        )
        
        net.from_nx(self.G)

        options = {
            "edges": {
                "color": {"inherit": "from", "opacity": 0.5},
                "smooth": {"type": "continuous", "roundness": 0.5},
                "arrows": {"to": {"enabled": True, "scaleFactor": 0.5}},
                "width": 1.5
            },
            "physics": {
                "forceAtlas2Based": {
                    "gravitationalConstant": -60,
                    "centralGravity": 0.005,
                    "springLength": 150,
                    "springStrength": 0.08,
                    "damping": 0.4,
                    "avoidOverlap": 0.5
                },
                "maxVelocity": 50,
                "minVelocity": 0.1,
                "solver": "forceAtlas2Based",
                "stabilization": {"enabled": True, "iterations": 1000}
            },
            "interaction": {
                "hover": True,
                "navigationButtons": True,
                "multiselect": True,
                "tooltipDelay": 200
            }
        }
        
        net.set_options(json.dumps(options))
        
        return net.write_html(filename)


In [11]:
builder = BuildGraph(files)
builder.build()
builder.show()

In [34]:
builder.push_to_neo4j("bolt://localhost:7687", "neo4j", "pk142145")

In [ ]:
import chromadb
from openai import OpenAI
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv()
openai_key = os.getenv('OPENAI_API_KEY')

class VectorStore:
    def __init__(self, files:dict, path='../chromadb', collection_name:str='database'):
        self.files = files
        self.client = chromadb.PersistentClient(path=path)
        ef = OpenAIEmbeddingFunction(api_key=openai_key, model_name="text-embedding-3-small")
        self.collection = self.client.get_or_create_collection(name=collection_name, embedding_function=ef)

    def build(self, max_module_lines=100, overlap=10):
        self.chunks = []
        for file in self.files.keys():
            content = self.files[file]['content']
            lines = content.split('\n')
            classes = self.files[file]['classes']
            functions = self.files[file]['functions']

            for i in range(0, len(lines), max_module_lines):
                start = max(0, i - overlap)
                end = min(len(lines), i + overlap + max_module_lines)
                chunk_lines = lines[start : end]
                self.chunks.append([
                    "\n".join(chunk_lines),
                    {
                        "path": file,
                        "filename": file.split("/")[-1],
                        "function_name": "module_level_segment",
                        "class_name": "module_level",
                        "line_start": start + 1,
                        "line_end": end
                    }
                ])

            for func in functions:
                parent = 'module_level'
                for c in classes:
                    if func['line_start'] >= c['line_start'] and func['line_start'] <= c['line_end']:
                        parent = c['name']
                        break
                
                start = max(0, func['line_start'] - overlap)
                end = min(len(lines), func['line_end'] + overlap)
                chunk = "\n".join(lines[start:end])
                metadata = {
                    "path": file.replace(),
                    "filename": file.split("/")[-1],
                    "function_name": func['name'],
                    "class_name": parent,
                    "line_start": start + 1,
                    "line_end": end
                }
                self.chunks.append([chunk, metadata])
                self.push()

    def push(self):
        if not self.chunks:
            print("No chunks to push. Run build() first.")
            return

        documents = [c[0] for c in self.chunks]
        metadatas = [c[1] for c in self.chunks]

        ids = [f"{m['path']}:{m['function_name']}:{i}" for i, m in enumerate(metadatas)]

        self.collection.add(
            ids=ids,
            documents=documents,
            metadatas=metadatas
        )
        print(f"Successfully pushed {len(documents)} chunks to collection '{self.collection.name}'.")


    def search(self, query: str, top_k: int = 5):
        results = self.collection.query(
            query_texts=[query],
            n_results=top_k
        )
        return results

In [13]:
client = VectorStore(files, collection_name='prototype')
# client.build()

In [32]:
# client.push()

In [14]:
results = client.search(
    query="what is the function tutor is doing and how many files it is importing from"
)
# (results['ids'])


In [15]:
results

{'ids': [['src\\services\\tutor_service.py:__init__:76',
   'app.py:module_level_segment:1',
   'app.py:module_level_segment:0',
   'src\\services\\tutor_service.py:module_level_segment:71',
   'README.md:module_level_segment:3']],
 'embeddings': None,
 'documents': [['from src.services.tutor_lesson_utils import (\n    PROCEED_PROMPT,\n    format_equations_for_prompt,\n    is_new_topic_intent,\n    is_proceed_to_quiz,\n)\n\n\nclass TutorWorkflow:\n    def __init__(self, neo4j_conn: Neo4jConn):\n        self.neo4j_conn = neo4j_conn\n        self.workflow = self._build_workflow()\n        self.app = self.workflow.compile(checkpointer=MemorySaver())\n        self.llm = ChatGroq(\n            model="llama-3.3-70b-versatile", \n            temperature=0.3, \n            max_tokens=250\n        )\n        # Longer explanations for the teach-first phase\n        self.teach_llm = ChatGroq(\n            model="llama-3.3-70b-versatile",\n            temperature=0.35,\n            max_tokens=1000

## RAG Evaluation metrices

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal

groq_api_key = os.getenv('GROQ_API_KEY')

class CypherQuery(BaseModel):
    cypher: str = Field(..., description="Cypher query for retrieving relationships within the nodes")

class RouterDecision(BaseModel):
    decision: Literal['hybrid', 'graph'] = Field(..., description="Graph for only relationships between nodes/files and hybrid for both graph and vector similarity search")
    reasoning: str = Field(..., description="Brief reasoning behind the routing decision")
    confidence_score: float = Field(..., description="Confidence score of the decision")

class QueryEngine:
    def __init__(self, repo_id: str, db_client: VectorStore, uri:str, user:str, password:str, llm: ChatGroq):
        self.db_client = db_client
        self.repo_id = repo_id
        self.llm = llm
        self.graph_driver = GraphDatabase.driver(uri, auth=(user, password))

    def vector_search(self, query: str, filenames: list[str], top_k: int = 5):
        collection = self.db_client.collection
        
        return collection.query(
            query_texts=[query],
            n_results=top_k,
            where={"path": {"$in": filenames}}
        )

    def graph_search(self, query):
        structured_llm = self.llm.with_structured_output(CypherQuery)

        prompt = ChatPromptTemplate.from_messages([
            ("system", """
            You are a cypher query master. Generate a cypher query on the basis of given prompt
            strictly following the structured output schema.
             
            SCHEMA:
            - Central Node: Repo (property: repo_id)
            - Relationship: (:Repo)-[:CONTAINS]->(:File)
            - Relationship: (:File)-[:IMPORTS]->(:File)
            
            CRITICAL RULES:
            1. Your root node is 'Repo', not 'Repository'.
            2. The property name is 'repo_id', not 'id'.
            3. Start every query with: MATCH (r:Repo {{repo_id: '{repo_id}'}})
            
            Example:
            MATCH (r:Repo {{repo_id: '{repo_id}'}})-[:CONTAINS]->(f:File {{name: 'tutor_service.py'}})
            MATCH (f)-[:IMPORTS]->(imported)
            RETURN f.path, imported.path
            """),
            ("user", "Question: {query} \nRepo ID: {repo_id}")
        ])

        chain = prompt | structured_llm

        response = chain.invoke({
            "query": query,
            "repo_id": self.repo_id
        })

        with self.graph_driver.session() as session:
            result = session.run(response.cypher)
            output = [record.data() for record in result]

        return output
    
    def router(self, query: str) -> RouterDecision:
        router_llm = self.llm.with_structured_output(RouterDecision)
        router_prompt = ChatPromptTemplate.from_messages([
            ('system', """
                You are a query router for a code repository assistant.
                Classify the query into one of three types:

                - graph: query asks about file relationships, imports, dependencies, structure
                - hybrid: query asks about all, structure, logic, function behavior, implementation detailsand implementation

                Return your decision as structured output.
            """),
            ('user', "query: {query}")
        ])

        router_chain = router_prompt | router_llm

        response = router_chain.invoke({"query": query})

        return response
    
    def get_result(self, query: str, top_k: int):
        decision = self.router(query)

        route = decision.decision
        reasoning = decision.reasoning
        score = decision.confidence_score

        if route == "graph":
            return {
                "type": "graph",
                "graph": self.graph_search(query),
                "reasone": reasoning,
                "confidence": score
            }

        else:
            graph_data = self.graph_search(query)

            filenames = list(set([
                v
                for r in graph_data
                for v in r.values()
                if v is not None
            ]))

            return {
                "type": "hybrid",
                "graph": graph_data,
                "vector": self.vector_search(query, filenames=filenames, top_k=top_k),
                "reasone": reasoning,
                "confidence": score
            }
            
    def close(self):
        self.graph_driver.close()
    

In [35]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    max_tokens=200,
    api_key=groq_api_key
)

engine = QueryEngine(repo_id=filename, db_client=client, uri="bolt://localhost:7687", user="neo4j", password="pk142145", llm=llm)
query = "How is app.py working?"
result = engine.get_result(query, 6)

In [36]:
result

{'type': 'hybrid',
 'graph': [{'f.path': 'app.py', 'imported.path': 'src/models/concept.py'},
  {'f.path': 'app.py', 'imported.path': 'src/models/state.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/pdf_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/llm_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/graph_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/intent_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/planner_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/diagnoser_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/services/tutor_service.py'},
  {'f.path': 'app.py', 'imported.path': 'src/database/student_db.py'},
  {'f.path': 'app.py', 'imported.path': 'src/database/neo4j_conn.py'}],
 'vector': {'ids': [['app.py:module_level_segment:1',
    'app.py:module_level_segment:0',
    'app.py:module_level_segment:2']],
  'embeddings': None,
  'documents': [['\n      

In [44]:
import ragas

In [39]:
from typing import TypedDict

class QueryState(TypedDict):
    question: str
    vector_context: str
    graph_context: str
    agent: str
    answer: str
